# RAG in 3 Lines

SynapseKit's `RAGPipeline` hides the embedding, storage, retrieval, and generation steps behind three async calls. This notebook shows you the minimum viable pipeline and then extends it with streaming.

**What you'll build:** A question-answering pipeline that embeds documents, stores them in memory, and answers natural-language questions.

**Time:** ~5 min | **Difficulty:** Beginner

Guide: [RAG in 3 Lines](https://synapsekit.github.io/synapsekit-docs/docs/guides/rag/quickstart-3-lines)

## Prerequisites & Setup

You will need:
- An **OpenAI API key** — set `OPENAI_API_KEY` in the env cell below
- `synapsekit` installed (run the install cell)

`InMemoryVectorStore` requires no external service — perfect for getting started quickly.

In [ ]:
!pip install synapsekit -q

In [ ]:
import os

# Set your OpenAI API key before running the cells below
# os.environ['OPENAI_API_KEY'] = 'sk-...'

## Step 1: Create the Pipeline

`RAGPipeline` is the only object you need. Every component — LLM, embeddings, vector store — is injected so you can swap any one of them later without touching the rest of your code.

In [ ]:
from synapsekit import RAGPipeline
from synapsekit.llms.openai import OpenAILLM
from synapsekit.embeddings.openai import OpenAIEmbeddings
from synapsekit.vectorstores.memory import InMemoryVectorStore

# InMemoryVectorStore requires no external service, so there is nothing to
# set up or tear down — perfect for getting started quickly.
rag = RAGPipeline(
    llm=OpenAILLM(model="gpt-4o-mini"),
    embeddings=OpenAIEmbeddings(model="text-embedding-3-small"),
    vectorstore=InMemoryVectorStore(),
)

## Step 2: Add Documents

`aadd()` accepts plain strings, `Document` objects, or any iterable of either. Internally it batches the embedding calls to stay within the API rate limit.

In [ ]:
docs = [
    "SynapseKit is an async-first Python library for building LLM applications.",
    "RAGPipeline supports PDF, DOCX, web, CSV, and plain-text sources.",
    "All SynapseKit retrieval methods are prefixed with 'a' to signal they are async.",
]

# aadd() embeds each string and persists the vectors in the store.
# Calling it once at startup means queries have zero ingestion latency.
await rag.aadd(docs)
print("Documents added successfully.")

## Step 3: Query

`aquery()` returns a plain string. The default `k=4` retrieves the four most similar chunks; you can override this with `rag.aquery("...", k=8)`.

In [ ]:
# aquery() embeds the question, retrieves the top-k most relevant chunks,
# and passes them as context to the LLM in a single round-trip.
answer = await rag.aquery("What kinds of sources does RAGPipeline support?")
print(answer)

## Step 4: Stream the Answer

`astream()` is an async generator that yields tokens as soon as the LLM produces them. The underlying retrieval step still happens once before generation starts.

In [ ]:
# Streaming lets you display partial output immediately instead of waiting
# for the full response — critical for good UX in chat interfaces.
async for token in rag.astream("What makes SynapseKit async-first?"):
    print(token, end="", flush=True)
print()  # newline after the stream finishes

## Step 5: Control How Many Chunks Are Retrieved

Raising `k` improves recall at the cost of a larger prompt. Lower `k` is faster and cheaper but may miss relevant context.

In [ ]:
# Raising k improves recall at the cost of a larger prompt.
# Lower k is faster and cheaper but may miss relevant context.
answer = await rag.aquery("What is SynapseKit?", k=2)
print(answer)

## Complete Working Example

A single self-contained cell that runs the entire pipeline end-to-end. Use `asyncio.run(main())` so it also works when run as a plain Python script.

In [ ]:
import asyncio
from synapsekit import RAGPipeline
from synapsekit.llms.openai import OpenAILLM
from synapsekit.embeddings.openai import OpenAIEmbeddings
from synapsekit.vectorstores.memory import InMemoryVectorStore

rag = RAGPipeline(
    llm=OpenAILLM(model="gpt-4o-mini"),
    embeddings=OpenAIEmbeddings(model="text-embedding-3-small"),
    vectorstore=InMemoryVectorStore(),
)

async def main():
    docs = [
        "SynapseKit is an async-first Python library for building LLM applications.",
        "RAGPipeline supports PDF, DOCX, web, CSV, and plain-text sources.",
        "All SynapseKit retrieval methods are prefixed with 'a' to signal they are async.",
    ]
    await rag.aadd(docs)

    answer = await rag.aquery("What kinds of sources does RAGPipeline support?")
    print("Answer:", answer)

    print("\nStreaming: ", end="")
    async for token in rag.astream("What makes SynapseKit async-first?"):
        print(token, end="", flush=True)
    print()

asyncio.run(main())

## Next Steps

- [Build a PDF Knowledge Base](https://synapsekit.github.io/synapsekit-docs/docs/guides/rag/pdf-knowledge-base) — ingest a real PDF file
- [Streaming RAG Responses](https://synapsekit.github.io/synapsekit-docs/docs/guides/rag/streaming-rag) — wire streaming into a FastAPI endpoint
- [RAGPipeline reference](https://synapsekit.github.io/synapsekit-docs/docs/rag/pipeline) — full API documentation